# Introduction

This notebook focuses on natural language understanding. Instead of experimenting with traditional models such as logistic regression or random forest, I chose to directly apply pre-trained models designed for language recognition. As I'm still at an early stage of learning in this field, I currently lack the experience to evaluate which models perform best in specific scenarios. Given the availability of well-developed open-source models, using them directly is a more practical and effective approach for me at this stage.

# Scenario

All the information you should know is [here](https://www.kaggle.com/competitions/nlp-getting-started).

# Data Exploration and Analysis

In addition to the natural language text, the dataset also includes two additional fields: keyword and location. Depending on the model used, these fields can be combined with the main text input to provide more context for the model to analyze. In other words, the provided data does not require much preprocessing at this stage.

Although several models were considered, this notebook ultimately uses microsoft/deberta-v3-base. After experimenting with adding location and keyword into the text input, I found that the performance actually dropped. As a result, I decided to remove these two fields and keep the rest of the data unchanged.

In [ ]:
import pandas as pd

In [ ]:
df_train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
df_raw = pd.concat([df_train,df_test],axis = 0)
df_raw_clean = df_raw[['id','text', 'target']]

# Model Building

The model used in this notebook is microsoft/deberta-v3-base, a transformer-based architecture from Microsoft's DeBERTa (Decoding-enhanced BERT with disentangled attention) series. Compared to standard BERT models, DeBERTa introduces two key improvements:

* Disentangled Attention Mechanism: It separates content and position information, allowing the model to better capture word relationships.

* Enhanced Positional Encoding: It improves how the model understands word order in a sentence.

In other words, I directly chose a relatively advanced model. And I have not fine-tuned its hyperparameters yet.

In [ ]:
# 按照 target 是否为空分成训练集和测试集
df_train_final = df_raw_clean[df_raw_clean['target'].notna()].copy()
df_test_final = df_raw_clean[df_raw_clean['target'].isna()].copy()
df_train_final['target'] = df_train_final['target'].astype(int)

1. Import Libraries
Necessary libraries are imported, including transformers, datasets, and sklearn.

2. Prepare the Data
The training and test data are extracted from preprocessed DataFrames. Targets are cast to integers.

3. Train-Validation Split
The training set is split into training and validation subsets using an 80/20 ratio.

4. Tokenization
The DeBERTa tokenizer (microsoft/deberta-v3-base) is loaded, and text data is tokenized for training, validation, and test sets.

5. Load the Pretrained Model
The DeBERTa model is loaded for binary classification (num_labels=2).

6. Set Training Arguments
Training configurations are specified, such as batch size, learning rate, number of epochs, and output settings.

7. Initialize the Trainer
The Trainer class is used to manage the training loop, evaluation, and data collation.

8. Model Training
The model is trained using the prepared datasets.

9. Prediction on Test Set
The trained model predicts labels for the test set, and the predicted logits are converted to class labels.

10. Save Submission File
Predictions are saved in submission.csv, and all other files are removed to keep only the submission output.

In [ ]:
# ============================
# 1. 导入依赖
# ============================
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import torch
import os

# ============================
# 2. 准备数据
# ============================
train_df = df_train_final[['text', 'target']].copy()
train_df['target'] = train_df['target'].astype(int)
test_df = df_test_final[['id', 'text']].copy()

# ============================
# 3. 划分验证集
# ============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['target'].tolist(),
    test_size=0.2,
    random_state=42
)

# ============================
# 4. 加载 tokenizer 并编码
# ============================
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})
test_dataset = Dataset.from_dict({"text": test_df['text'].tolist()})

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# ============================
# 5. 加载 DeBERTa 模型
# ============================
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)

# ============================
# 6. 设置训练参数
# ============================
training_args = TrainingArguments(
    output_dir="./tmp",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    save_strategy="no",        # 不保存模型
    disable_tqdm=False
)

# ============================
# 7. 设置 Trainer
# ============================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ============================
# 8. 开始训练
# ============================
trainer.train()

# ============================
# 9. 预测测试集
# ============================
predictions = trainer.predict(test_dataset)
pred_labels = torch.argmax(torch.tensor(predictions.predictions), axis=1).numpy()

# ============================
# 10. 保存提交文件
# ============================
test_df['target'] = pred_labels
test_df[['id', 'target']].to_csv("submission.csv", index=False)

# ✅ 清理其他输出，只保留提交文件
for f in os.listdir():
    if f != "submission.csv":
        if os.path.isdir(f):
            os.system(f"rm -rf {f}")
        else:
            os.remove(f)

print("✅ submission.csv 已生成，并已清理其他文件。")
# best

# Conclusion

Natural language processing can be approached in different ways. One option is to use traditional models by processing the data manually, tuning hyperparameters, and gradually improving performance. Another option is to apply more advanced pre-trained models to achieve strong results more directly.

That said, traditional models still have their value. While NLP techniques continue to evolve, using the latest models is not always the most cost-effective solution, especially when considering factors such as computational resources, training time, and deployment complexity. Older models are often more stable, and in well-established industries, switching to newer architectures may introduce uncertainties that many organizations prefer to avoid.

In addition, simpler models are generally easier to understand and work with. For beginners like myself, starting with more basic models can be helpful for building a deeper understanding of the mathematical foundations behind them.

Still, in this notebook, I chose to use a relatively advanced model for my current level and went straight to generating results.